# Predictor de resultados de Premier League

Cuaderno de investigación y evaluación de cinco especificaciones de regresión de Poisson. El flujo está organizado para ejecutarse secuencialmente, desde la base histórica hasta una predicción individual. El entrenamiento utiliza únicamente partidos anteriores a cada observación; las cuotas se usan como referencia de evaluación, no como predictores.

**Archivos requeridos en el directorio de trabajo:** `wc_predictor.py` y `E0_consolidado.csv`. El archivo `premier_training_data.csv` es una caché opcional de variables históricas; debe regenerarse si cambian la fuente de datos o las funciones de construcción de variables. Si `wc_predictor.load_history()` usa un CSV diferente de `E0_consolidado.csv`, unifique las fuentes antes de ejecutar.

El archivo original `Prueba.ipynb` se conserva sin modificaciones. Esta versión organiza los mismos modelos M0–M4, las métricas MAE y Log-Loss, las correlaciones/VIF, la comparación contra cuotas de apertura y una predicción individual. No corrige por sí sola las limitaciones metodológicas que puedan existir dentro de `wc_predictor.py`.

## 1. Dependencias y configuración

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import poisson, skellam
from sklearn.metrics import log_loss, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

import wc_predictor

# Fechas inclusivas por la izquierda: [inicio, fin)
FECHA_INICIO = pd.Timestamp("2019-08-01")
FECHA_VALIDACION = pd.Timestamp("2024-08-01")
FECHA_PRUEBA = pd.Timestamp("2025-08-01")

K_SHRINKAGE = 10
N_FORMA = 10
DECAY_FORMA = 0.85
ESCALA_ELO = 400

ARCHIVO_CUOTAS = Path("E0_consolidado.csv")
ARCHIVO_VARIABLES = Path("premier_training_data.csv")
RECONSTRUIR_VARIABLES = False  # Cambiar a true si cambió el histórico

CLAVES = ["Date", "HomeTeam", "AwayTeam"]
OBJETIVOS = ["home_goals", "away_goals"]
CUOTAS = ["AvgH", "AvgD", "AvgA"]
PROBS = ["P_home", "P_draw", "P_away"]

GRUPOS = {
    "base_home": ["elo_diff", "gf_home", "ga_away"],
    "base_away": ["elo_diff", "gf_away", "ga_home"],
    "forma_home": ["form_gf_home", "form_ga_away"],
    "forma_away": ["form_gf_away", "form_ga_home"],
    "tiros_home": ["shots_for_home", "shots_against_away"],
    "tiros_away": ["shots_for_away", "shots_against_home"],
    "sot_home": ["sot_for_home", "sot_against_away"],
    "sot_away": ["sot_for_away", "sot_against_home"],
}

ESPECIFICACIONES = {
    "M0_Base": (GRUPOS["base_home"], GRUPOS["base_away"]),
    "M1_Forma": (
        GRUPOS["base_home"] + GRUPOS["forma_home"],
        GRUPOS["base_away"] + GRUPOS["forma_away"],
    ),
    "M2_Tiros": (
        GRUPOS["base_home"] + GRUPOS["tiros_home"],
        GRUPOS["base_away"] + GRUPOS["tiros_away"],
    ),
    "M3_SOT": (
        GRUPOS["base_home"] + GRUPOS["sot_home"],
        GRUPOS["base_away"] + GRUPOS["sot_away"],
    ),
    "M4_Completo": (
        GRUPOS["base_home"] + GRUPOS["forma_home"] + GRUPOS["tiros_home"] + GRUPOS["sot_home"],
        GRUPOS["base_away"] + GRUPOS["forma_away"] + GRUPOS["tiros_away"] + GRUPOS["sot_away"],
    ),
}

TODOS_LOS_PREDICTORES = sorted({
    columna
    for par in ESPECIFICACIONES.values()
    for columnas in par
    for columna in columnas
})
COLUMNAS_TIROS = [c for c in TODOS_LOS_PREDICTORES if c.startswith(("shots_", "sot_"))]

Arsenal                   1833.29
Man City                  1807.88
Man United                1679.95
Bournemouth               1659.96
Liverpool                 1656.03
Brighton                  1625.52
Brentford                 1624.68
Aston Villa               1618.60
Leeds                     1612.92
Chelsea                   1612.18
Nott'm Forest             1595.24
Newcastle                 1591.52
Everton                   1590.77
Sunderland                1565.71
Fulham                    1552.14
West Ham                  1549.48
Crystal Palace            1538.37
Tottenham                 1502.05
Hull                      1497.31
Wigan                     1490.64
Stoke                     1477.24
Birmingham                1473.32
Bolton                    1470.11
Blackpool                 1469.01
Coventry                  1466.05
Swansea                   1463.15
Charlton                  1457.07
Wolves                    1449.40
Cardiff                   1433.82
Blackburn     

## 2. Construcción uniforme de variables previas al partido

Una misma función se usa para generar las observaciones de entrenamiento y las de partidos nuevos. `elo_diff` se guarda en puntos Elo y se divide entre 400 solamente al construir la matriz de regresores. Se excluye siempre el partido que se desea predecir.

In [2]:
def crear_variables_partido(df_pre, fecha, home, away, elo_home, elo_away):
    """Devuelve predictores construidos sólo con partidos anteriores a `fecha`.

    `df_pre` debe contener exclusivamente partidos con date < fecha.
    Los ratings Elo corresponden al instante anterior al encuentro.
    """
    fecha = pd.Timestamp(fecha)
    if not (df_pre["date"] < fecha).all():
        raise ValueError("df_pre contiene información de la fecha objetivo o posterior")

    stats_h = wc_predictor.season_stats(df_pre, home, fecha, k=K_SHRINKAGE)
    stats_a = wc_predictor.season_stats(df_pre, away, fecha, k=K_SHRINKAGE)
    form_h = wc_predictor.recent_form(df_pre, home, n=N_FORMA, decay=DECAY_FORMA)
    form_a = wc_predictor.recent_form(df_pre, away, n=N_FORMA, decay=DECAY_FORMA)

    return {
        "elo_home": elo_home, "elo_away": elo_away,
        "elo_diff": elo_home - elo_away,
        "gf_home": stats_h["gf_avg"], "ga_home": stats_h["ga_avg"],
        "gf_away": stats_a["gf_avg"], "ga_away": stats_a["ga_avg"],
        "form_gf_home": form_h["gf"], "form_ga_home": form_h["ga"],
        "form_gf_away": form_a["gf"], "form_ga_away": form_a["ga"],
        "shots_for_home": form_h["shots_for"],
        "shots_against_home": form_h["shots_against"],
        "shots_for_away": form_a["shots_for"],
        "shots_against_away": form_a["shots_against"],
        "sot_for_home": form_h["sot_for"],
        "sot_against_home": form_h["sot_against"],
        "sot_for_away": form_a["sot_for"],
        "sot_against_away": form_a["sot_against"],
    }


def construir_base_historica(historial, desde=FECHA_INICIO):
    """Calcula predictores prepartido y actualiza Elo después de cada fecha.

    Los partidos del mismo día usan el mismo corte histórico. Para reflejar
    el experimento original, Elo se actualiza una vez registrados todos ellos.
    """
    historial = historial.sort_values("date", kind="stable").reset_index(drop=True)
    ratings = {}
    registros = []

    for fecha, partidos_dia in historial.groupby("date", sort=True):
        df_pre = historial.loc[historial["date"] < fecha]
        elo_previo = ratings.copy()

        if fecha >= desde:
            for partido in partidos_dia.itertuples(index=False):
                home, away = partido.home_team, partido.away_team
                variables = crear_variables_partido(
                    df_pre, fecha, home, away,
                    elo_previo.get(home, wc_predictor.ELO_INIT),
                    elo_previo.get(away, wc_predictor.ELO_INIT),
                )
                registros.append({
                    "Date": fecha, "HomeTeam": home, "AwayTeam": away,
                    **variables,
                    "home_goals": partido.home_score,
                    "away_goals": partido.away_score,
                })

        for partido in partidos_dia.itertuples(index=False):
            home, away = partido.home_team, partido.away_team
            rh = ratings.get(home, wc_predictor.ELO_INIT)
            ra = ratings.get(away, wc_predictor.ELO_INIT)
            sh = 1.0 if partido.home_score > partido.away_score else (
                0.0 if partido.home_score < partido.away_score else 0.5
            )
            ratings[home] = wc_predictor.update_elo(rh, ra, sh)
            ratings[away] = wc_predictor.update_elo(ra, rh, 1.0 - sh)

    return pd.DataFrame(registros)

## 3. Carga, control de calidad y partición cronológica

La caché permite repetir los análisis sin reconstruir miles de filas. Para reproducir la ingeniería de variables desde la fuente, establezca `RECONSTRUIR_VARIABLES = True`. El histórico y el módulo deben corresponder a la misma versión usada al generar la caché.

In [3]:
historial = wc_predictor.load_history()
historial["date"] = pd.to_datetime(historial["date"])

if RECONSTRUIR_VARIABLES or not ARCHIVO_VARIABLES.exists():
    training_data = construir_base_historica(historial)
    n_antes = len(training_data)
    training_data = training_data.dropna(subset=COLUMNAS_TIROS).reset_index(drop=True)
    training_data.to_csv(ARCHIVO_VARIABLES, index=False)
    print(f"Base regenerada: {n_antes} partidos, {len(training_data)} tras limpieza")
else:
    training_data = pd.read_csv(ARCHIVO_VARIABLES)
    print(f"Base cargada desde {ARCHIVO_VARIABLES} ({len(training_data)} partidos)")

training_data["Date"] = pd.to_datetime(training_data["Date"])
training_data = training_data.sort_values("Date", kind="stable").reset_index(drop=True)

faltantes = training_data[TODOS_LOS_PREDICTORES + OBJETIVOS].apply(
    pd.to_numeric, errors="coerce"
)
if not np.isfinite(faltantes.to_numpy(dtype=float)).all():
    raise ValueError("Existen variables no numéricas, faltantes o infinitas. Revise la base histórica.")
if training_data.duplicated(CLAVES).any():
    raise ValueError("Hay partidos duplicados según fecha y equipos.")

train = training_data.loc[training_data["Date"] < FECHA_VALIDACION].copy()
val = training_data.loc[
    (training_data["Date"] >= FECHA_VALIDACION)
    & (training_data["Date"] < FECHA_PRUEBA)
].copy()
test = training_data.loc[training_data["Date"] >= FECHA_PRUEBA].copy()

for nombre, datos in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    if datos.empty:
        raise ValueError(f"El conjunto {nombre} está vacío")
    print(f"{nombre}: {len(datos)} partidos, {datos['Date'].min().date()} a {datos['Date'].max().date()}")

Base cargada desde premier_training_data.csv (2696 partidos)
Entrenamiento: 1897 partidos, 2019-08-09 a 2024-05-19
Validación: 380 partidos, 2024-08-16 a 2025-05-25
Prueba: 419 partidos, 2025-08-15 a 2026-09-14


## 4. Funciones estadísticas compartidas

La división Elo/400, el intercepto y el orden de predictores se resuelven en un solo lugar. La función de evaluación devuelve tanto las métricas agregadas como las probabilidades por partido; estas últimas se reutilizan en la comparación con el mercado.

In [4]:
def preparar_X(datos, columnas, modelo=None):
    """Prepara regresores, incluida la escala de Elo y el intercepto."""
    X = datos.loc[:, columnas].copy().astype(float)
    X["elo_diff"] = X["elo_diff"] / ESCALA_ELO
    X = sm.add_constant(X, has_constant="add")
    if modelo is not None:
        X = X.loc[:, modelo.model.exog_names]
    return X


def entrenar_modelo(datos, columnas_home, columnas_away):
    """Estima una regresión Poisson para goles locales y otra para visitantes."""
    home = sm.GLM(
        datos["home_goals"], preparar_X(datos, columnas_home),
        family=sm.families.Poisson(),
    ).fit()
    away = sm.GLM(
        datos["away_goals"], preparar_X(datos, columnas_away),
        family=sm.families.Poisson(),
    ).fit()
    return {"home": home, "away": away,
            "columnas_home": columnas_home, "columnas_away": columnas_away}


def probabilidades_1x2(lambda_home, lambda_away):
    """Probabilidades [victoria local, empate, victoria visitante]."""
    lh, la = np.asarray(lambda_home), np.asarray(lambda_away)
    probs = np.column_stack((
        skellam.sf(0, lh, la),
        skellam.pmf(0, lh, la),
        skellam.cdf(-1, lh, la),
    ))
    if not np.isfinite(probs).all() or not np.allclose(probs.sum(axis=1), 1.0, atol=1e-8):
        raise ValueError("Las probabilidades 1X2 no son válidas")
    return probs


def resultados_observados(datos):
    """Codifica 0=local, 1=empate y 2=visitante."""
    gh = datos["home_goals"].to_numpy()
    ga = datos["away_goals"].to_numpy()
    return np.where(gh > ga, 0, np.where(gh == ga, 1, 2))


def predecir_con_modelo(modelo, datos):
    """Devuelve goles esperados y probabilidades para un conjunto de encuentros."""
    lh = np.asarray(modelo["home"].predict(
        preparar_X(datos, modelo["columnas_home"], modelo["home"])
    ), dtype=float)
    la = np.asarray(modelo["away"].predict(
        preparar_X(datos, modelo["columnas_away"], modelo["away"])
    ), dtype=float)
    if not np.isfinite(lh).all() or not np.isfinite(la).all():
        raise ValueError("Se obtuvieron goles esperados no finitos")
    probs = probabilidades_1x2(lh, la)
    resultado = datos[CLAVES + OBJETIVOS].reset_index(drop=True).copy() if all(
        c in datos for c in OBJETIVOS
    ) else datos[CLAVES].reset_index(drop=True).copy()
    resultado["lambda_home"] = lh
    resultado["lambda_away"] = la
    resultado[PROBS] = probs
    return resultado


def evaluar_predicciones(pred):
    """Métricas para una tabla con goles reales y predicciones."""
    mae_h = mean_absolute_error(pred["home_goals"], pred["lambda_home"])
    mae_a = mean_absolute_error(pred["away_goals"], pred["lambda_away"])
    return {
        "Partidos": len(pred),
        "MAE_local": mae_h,
        "MAE_visitante": mae_a,
        "MAE_promedio": (mae_h + mae_a) / 2,
        "LogLoss_1X2": log_loss(
            resultados_observados(pred), pred[PROBS].to_numpy(), labels=[0, 1, 2]
        ),
    }

In [10]:
print("MODELO BASE M0 — LOCAL")
print(modelos_entrenados["M0_Base"]["home"].summary())

print("\nMODELO BASE M0 — VISITANTE")
print(modelos_entrenados["M0_Base"]["away"].summary())

MODELO BASE M0 — LOCAL
                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_goals   No. Observations:                 1897
Model:                            GLM   Df Residuals:                     1893
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2888.2
Date:                Sun, 20 Sep 2026   Deviance:                       2122.6
Time:                        22:54:21   Pearson chi2:                 1.88e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1557
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3864      0.

## 5. Estimación M0–M4 y validación 2024/25

La elección exploratoria de especificaciones utiliza sólo el periodo de validación; el conjunto de prueba permanece aparte hasta la sección 8.

In [5]:
modelos_entrenados = {
    nombre: entrenar_modelo(train, *columnas)
    for nombre, columnas in ESPECIFICACIONES.items()
}

predicciones_val = {
    nombre: predecir_con_modelo(modelo, val)
    for nombre, modelo in modelos_entrenados.items()
}

tabla_validacion = pd.DataFrame([
    {"Modelo": nombre, "Variables": len(modelos_entrenados[nombre]["columnas_home"]),
     **evaluar_predicciones(pred)}
    for nombre, pred in predicciones_val.items()
])

# Diferencia contra la versión M0 evaluada con las mismas observaciones
ll_m0 = tabla_validacion.loc[
    tabla_validacion["Modelo"] == "M0_Base", "LogLoss_1X2"
].iloc[0]
tabla_validacion["Delta_LogLoss_vs_M0"] = tabla_validacion["LogLoss_1X2"] - ll_m0

display(tabla_validacion.sort_values("LogLoss_1X2").round(6))

,Modelo,Variables,Partidos,MAE_local,MAE_visitante,MAE_promedio,LogLoss_1X2,Delta_LogLoss_vs_M0
4,M4_Completo,9,380,0.977467,0.872569,0.925018,0.975296,-0.008394
2,M2_Tiros,5,380,0.978974,0.870019,0.924496,0.977038,-0.006652
3,M3_SOT,5,380,0.977342,0.872963,0.925152,0.977225,-0.006465
1,M1_Forma,5,380,0.981990,0.873843,0.927916,0.982334,-0.001356
0,M0_Base,3,380,0.982484,0.872901,0.927692,0.983690,0.000000


## 6. Modelo de referencia simple y diagnóstico del modelo ampliado

El modelo de referencia utiliza las medias de goles calculadas únicamente en entrenamiento. Las correlaciones y el VIF describen dependencias entre predictores; no son criterios automáticos para descartarlos.

In [6]:
lambda_base_home = float(train["home_goals"].mean())
lambda_base_away = float(train["away_goals"].mean())

base_val = val[CLAVES + OBJETIVOS].reset_index(drop=True).copy()
base_val["lambda_home"] = lambda_base_home
base_val["lambda_away"] = lambda_base_away
base_val[PROBS] = probabilidades_1x2(
    np.full(len(val), lambda_base_home),
    np.full(len(val), lambda_base_away),
)
print("Modelo de referencia simple:", evaluar_predicciones(base_val))


def tabla_vif(datos, columnas):
    """Factor de inflación de la varianza; excluye el intercepto."""
    X = preparar_X(datos, columnas)
    matriz = X.to_numpy(dtype=float)
    return pd.DataFrame({
        "Variable": X.columns[1:],
        "VIF": [variance_inflation_factor(matriz, i)
                for i in range(1, X.shape[1])],
    }).sort_values("VIF", ascending=False).reset_index(drop=True)

m4 = modelos_entrenados["M4_Completo"]
print("Modelo local M4")
print(m4["home"].summary())
print("Modelo visitante M4")
print(m4["away"].summary())

for lado in ("home", "away"):
    columnas = m4[f"columnas_{lado}"]
    print(f"Correlaciones M4: {lado}")
    display(train[columnas].corr().round(3))
    print(f"VIF M4: {lado}")
    display(tabla_vif(train, columnas).round(3))

Modelo de referencia simple: {'Partidos': 380, 'MAE_local': 1.054156146824626, 'MAE_visitante': 0.9281386122131897, 'MAE_promedio': 0.9911473795189079, 'LogLoss_1X2': 1.0793610651054353}
Modelo local M4
                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_goals   No. Observations:                 1897
Model:                            GLM   Df Residuals:                     1887
Model Family:                 Poisson   Df Model:                            9
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2876.3
Date:                Sun, 20 Sep 2026   Deviance:                       2098.8
Time:                        22:51:09   Pearson chi2:                 1.86e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1662
Covariance Type:            nonrobust                                         
       

,elo_diff,gf_home,ga_away,form_gf_home,form_ga_away,shots_for_home,shots_against_away,sot_for_home,sot_against_away
elo_diff,1.000,0.616,0.564,0.514,0.413,0.472,0.445,0.491,0.413
gf_home,0.616,1.000,0.015,0.807,-0.002,0.696,0.002,0.737,-0.000
ga_away,0.564,0.015,1.000,0.036,0.747,0.014,0.643,0.021,0.656
form_gf_home,0.514,0.807,0.036,1.000,0.046,0.660,0.025,0.787,0.029
form_ga_away,0.413,-0.002,0.747,0.046,1.000,0.027,0.586,0.025,0.726
shots_for_home,0.472,0.696,0.014,0.660,0.027,1.000,0.018,0.854,0.016
shots_against_away,0.445,0.002,0.643,0.025,0.586,0.018,1.000,0.020,0.818
sot_for_home,0.491,0.737,0.021,0.787,0.025,0.854,0.020,1.000,0.013
sot_against_away,0.413,-0.000,0.656,0.029,0.726,0.016,0.818,0.013,1.000


VIF M4: home


,Variable,VIF
0,sot_for_home,5.517
1,gf_home,4.422
2,sot_against_away,4.196
3,shots_for_home,3.985
4,form_gf_home,3.886
5,ga_away,3.450
6,shots_against_away,3.406
7,elo_diff,3.384
8,form_ga_away,3.017


Correlaciones M4: away


,elo_diff,gf_away,ga_home,form_gf_away,form_ga_home,shots_for_away,shots_against_home,sot_for_away,sot_against_home
elo_diff,1.000,-0.614,-0.570,-0.511,-0.407,-0.465,-0.451,-0.480,-0.410
gf_away,-0.614,1.000,0.026,0.814,0.005,0.693,0.019,0.734,0.007
ga_home,-0.570,0.026,1.000,0.019,0.754,0.029,0.649,0.032,0.667
form_gf_away,-0.511,0.814,0.019,1.000,0.011,0.654,0.033,0.784,0.012
form_ga_home,-0.407,0.005,0.754,0.011,1.000,0.050,0.586,0.032,0.725
shots_for_away,-0.465,0.693,0.029,0.654,0.050,1.000,0.041,0.848,0.027
shots_against_home,-0.451,0.019,0.649,0.033,0.586,0.041,1.000,0.040,0.823
sot_for_away,-0.480,0.734,0.032,0.784,0.032,0.848,0.040,1.000,0.017
sot_against_home,-0.410,0.007,0.667,0.012,0.725,0.027,0.823,0.017,1.000


VIF M4: away


,Variable,VIF
0,sot_for_away,5.281
1,gf_away,4.430
2,sot_against_home,4.298
3,form_gf_away,3.946
4,shots_for_away,3.851
5,ga_home,3.609
6,shots_against_home,3.490
7,elo_diff,3.323
8,form_ga_home,3.051


## 7. Cuotas de apertura como referencia

Se convierten las cuotas decimales a probabilidades y se normalizan por partido. Esta comparación es de calidad probabilística, no de rentabilidad económica; la disponibilidad de las cuotas en el instante de pronóstico debe verificarse por separado.

In [7]:
def cargar_cuotas(ruta):
    """Lee cuotas promedio de apertura y verifica identificadores únicos."""
    mercado = pd.read_csv(ruta, usecols=CLAVES + CUOTAS)
    mercado["Date"] = pd.to_datetime(mercado["Date"], dayfirst=True, format="mixed")
    mercado[CUOTAS] = mercado[CUOTAS].apply(pd.to_numeric, errors="coerce")
    if mercado.duplicated(CLAVES).any():
        raise ValueError("Hay registros duplicados en el archivo de cuotas")
    return mercado


def comparar_con_mercado(predicciones_por_modelo, mercado):
    """Evalúa mercado y modelos sobre el mismo subconjunto con cuotas válidas."""
    modelo_referencia = next(iter(predicciones_por_modelo))
    base = predicciones_por_modelo[modelo_referencia][CLAVES + OBJETIVOS]
    datos = base.merge(mercado, on=CLAVES, how="left", validate="one_to_one")
    cuotas = datos[CUOTAS].to_numpy(dtype=float)
    validas = np.isfinite(cuotas).all(axis=1) & (cuotas > 1).all(axis=1)
    datos = datos.loc[validas].reset_index(drop=True)
    if datos.empty:
        raise ValueError("No hay encuentros con cuotas válidas")
    brutas = 1.0 / datos[CUOTAS].to_numpy(dtype=float)
    totales = brutas.sum(axis=1)
    prob_mercado = brutas / totales[:, None]
    y = resultados_observados(datos)
    tabla = [{"Modelo": "Mercado_apertura", "Partidos": len(datos),
              "LogLoss_1X2": log_loss(y, prob_mercado, labels=[0, 1, 2])}]
    for nombre, pred in predicciones_por_modelo.items():
        alineado = datos[CLAVES].merge(
            pred[CLAVES + PROBS], on=CLAVES, how="left", validate="one_to_one"
        )
        prob = alineado[PROBS].to_numpy(dtype=float)
        if not np.isfinite(prob).all() or not np.allclose(prob.sum(axis=1), 1.0, atol=1e-8):
            raise ValueError(f"Probabilidades inválidas para {nombre}")
        tabla.append({"Modelo": nombre, "Partidos": len(datos),
                      "LogLoss_1X2": log_loss(y, prob, labels=[0, 1, 2])})
    return (pd.DataFrame(tabla).sort_values("LogLoss_1X2").reset_index(drop=True),
            len(base), len(datos), (totales.mean() - 1.0) * 100)

mercado = cargar_cuotas(ARCHIVO_CUOTAS)
tabla_mercado_val, n_val, n_val_cuotas, margen_val = comparar_con_mercado(
    predicciones_val, mercado
)
print(f"Validación: {n_val_cuotas}/{n_val} con cuotas válidas; margen bruto promedio {margen_val:.2f}%")
display(tabla_mercado_val.round(6))

Validación: 380/380 con cuotas válidas; margen bruto promedio 4.49%


,Modelo,Partidos,LogLoss_1X2
0,Mercado_apertura,380,0.970552
1,M4_Completo,380,0.975296
2,M2_Tiros,380,0.977038
3,M3_SOT,380,0.977225
4,M1_Forma,380,0.982334
5,M0_Base,380,0.983690


## 8. Evaluación final en prueba y comparación con el mercado

Los modelos ya están entrenados. En esta sección no se recalculan coeficientes ni se redefine ninguna especificación en función de los resultados de prueba.

In [8]:
MODELOS_PRUEBA = ("M0_Base", "M2_Tiros", "M4_Completo")

predicciones_test = {
    nombre: predecir_con_modelo(modelos_entrenados[nombre], test)
    for nombre in MODELOS_PRUEBA
}

tabla_test = pd.DataFrame([
    {"Modelo": nombre, **evaluar_predicciones(pred)}
    for nombre, pred in predicciones_test.items()
])
print(f"Prueba: {len(test)} partidos; {test['Date'].min().date()} a {test['Date'].max().date()}")
display(tabla_test.round(6))

tabla_mercado_test, n_test, n_test_cuotas, margen_test = comparar_con_mercado(
    predicciones_test, mercado
)
print(f"Cuotas válidas en prueba: {n_test_cuotas}/{n_test}; margen bruto promedio: {margen_test:.2f}%")
display(tabla_mercado_test.round(6))

Prueba: 419 partidos; 2025-08-15 a 2026-09-14


,Modelo,Partidos,MAE_local,MAE_visitante,MAE_promedio,LogLoss_1X2
0,M0_Base,419,0.954649,0.847060,0.900854,1.030556
1,M2_Tiros,419,0.945192,0.855826,0.900509,1.035184
2,M4_Completo,419,0.940582,0.854688,0.897635,1.034434


Cuotas válidas en prueba: 419/419; margen bruto promedio: 5.84%


,Modelo,Partidos,LogLoss_1X2
0,Mercado_apertura,419,1.020000
1,M0_Base,419,1.030556
2,M4_Completo,419,1.034434
3,M2_Tiros,419,1.035184


## 9. Predicción individual

La función usa la misma ingeniería de variables que la base de entrenamiento. El argumento `fecha` establece el corte: sólo se consideran los partidos anteriores a esa fecha. Los nombres de equipo deben coincidir con los del histórico.

In [9]:
def predecir_partido(home, away, fecha, nombre_modelo="M4_Completo"):
    """Pronostica goles, probabilidades 1X2 y marcador modal para un encuentro."""
    fecha = pd.Timestamp(fecha)
    df_pre = historial.loc[historial["date"] < fecha].copy()
    if df_pre.empty:
        raise ValueError("No hay histórico anterior a la fecha indicada")
    equipos = set(df_pre["home_team"]) | set(df_pre["away_team"])
    if home not in equipos or away not in equipos:
        raise ValueError("Uno o ambos equipos no aparecen en el histórico anterior a la fecha")

    elo = wc_predictor.build_elo(df_pre)
    fila = {
        "Date": fecha, "HomeTeam": home, "AwayTeam": away,
        **crear_variables_partido(
            df_pre, fecha, home, away,
            elo.get(home, wc_predictor.ELO_INIT),
            elo.get(away, wc_predictor.ELO_INIT),
        ),
    }
    datos = pd.DataFrame([fila])
    modelo = modelos_entrenados[nombre_modelo]
    pred = predecir_con_modelo(modelo, datos).iloc[0]

    goles = np.arange(11)
    matriz = np.outer(
        poisson.pmf(goles, pred["lambda_home"]),
        poisson.pmf(goles, pred["lambda_away"]),
    )
    gh, ga = np.unravel_index(matriz.argmax(), matriz.shape)
    return {
        "fecha_corte": fecha.date().isoformat(),
        "partido": f"{home} vs {away}",
        "modelo": nombre_modelo,
        "lambda_home": float(pred["lambda_home"]),
        "lambda_away": float(pred["lambda_away"]),
        "P_home": float(pred["P_home"]),
        "P_draw": float(pred["P_draw"]),
        "P_away": float(pred["P_away"]),
        "marcador_mas_probable": f"{gh}-{ga}",
        "P_marcador": float(matriz[gh, ga]),
    }

# Ejemplo hipotético. Cambie equipos, fecha y especificación según corresponda.
ejemplo = predecir_partido("Arsenal", "Man City", "2026-09-20", "M4_Completo")
for nombre, valor in ejemplo.items():
    print(f"{nombre}: {valor}")

fecha_corte: 2026-09-20
partido: Arsenal vs Man City
modelo: M4_Completo
lambda_home: 1.562028793332503
lambda_away: 1.2113198583075406
P_home: 0.4545826233060055
P_draw: 0.2497686750402626
P_away: 0.29564870165373197
marcador_mas_probable: 1-1
P_marcador: 0.118167447991293


## 10. Alcance y reproducibilidad

- El entrenamiento termina antes de 2024-08-01; la validación comprende 2024/25; la prueba inicia en 2025-08-01. La comparación del mercado se realiza sobre el mismo subconjunto de encuentros con cuotas válidas.
- La independencia condicional de los goles bajo el modelo Poisson/Skellam es una hipótesis de modelación. El Log-Loss 1X2 y el MAE miden objetivos distintos.
- La caché de variables debe regenerarse cuando cambie el histórico, wc_predictor.py o los hiperparámetros. Si modifica el módulo en Jupyter, reinicie el kernel y ejecute el cuaderno desde el principio.
- No se realizó una evaluación de rentabilidad económica. Las cuotas de apertura son sólo una referencia probabilística.